# imports 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType,DateType

# read date from Bronze Layer

In [0]:
df = spark.read.table('`e-commerce-databricks-project`.bronze_layer.crm_prd_info')


# apply Data Transformations for silver Table

In [0]:
df.display()

## Renaming columns

In [0]:

RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old,new in RENAME_MAP.items():
    df = df.withColumnRenamed(old,new)

## Remove Records with Missing Product ID

In [0]:
df = df.filter(col('product_id').isNotNull())

## Triming String columns

In [0]:
for colName, colType in df.dtypes:
    if colType == 'string':
        df = df.withColumn(colName,trim(col(colName)))
df.display()

## Data Normalization for prd_line

In [0]:
df = (
    df.withColumn(
        "product_line",
         F.when(upper(col("product_line")) == "M", "Mountain")
         .when(upper(col("product_line")) == "R", "Road")
         .when(upper(col("product_line")) == "S", "Other Sales")
         .when(upper(col("product_line")) == "T", "Touring")
         .otherwise("Unknown")
    )
)
df.display()

## Cost Cleanup

In [0]:
df = df.withColumn('product_cost', coalesce(col('product_cost'), lit(0)))

## fixing start_date


In [0]:
df = df.withColumn("start_date", to_date(col("start_date"), "M/d/yyyy"))
df.display()


In [0]:
df = df.withColumn(
    "category_id",translate(F.substring(col("product_number"), 1, 5), "-", "_"))
df = df.withColumn("product_number", substring(col("product_number"), 7, length(col("product_number"))))

## Final Look

In [0]:
df.display()

# write data into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('`e-commerce-databricks-project`.silver_layer.product_info')